# 09 — Build Ground Truth (LLM-as-a-Judge Evaluation Flow)
**Project:** Semantic Book Recommender — IT4142 HUST  
**Input:** `data/processed/books_clean.csv`  
**Output:** `data/eval/qrels.json` (TREC-style ground truth query-to-document relevance map)  

Notebook này chứa logic tạo bộ dữ liệu đánh giá chất lượng cao (Ground Truth) sử dụng phương pháp **LLM-as-a-Judge** tích hợp từ mã nguồn hệ thống mới (`src/` modules):
1. **LLM**: Sử dụng OpenAI (`gpt-4o-mini`) để sinh câu hỏi tự động và đánh giá độ liên quan.
2. **Cơ chế**: Không chỉ map 1-1 đơn giản, mà quét qua một nhóm ứng viên (candidate pool) và nhờ LLM chấm điểm độ liên quan ở 3 mức độ (0: không liên quan, 1: liên quan một phần, 2: rất liên quan) để tạo file `qrels.json` chuẩn công nghiệp.

## 1. Khởi tạo cấu hình và Load dữ liệu sách

In [1]:
import os
from pathlib import Path
import sys

# Auto-adjust working directory to 'AI' if running from the root workspace
cwd = Path(os.getcwd())
if (cwd / 'AI').exists() and not (cwd / 'src').exists():
    os.chdir(cwd / 'AI')
    print(f"Adjusted working directory to: {os.getcwd()}")
else:
    print(f"Current working directory: {os.getcwd()}")

# Ensure src is importable
sys.path.insert(0, str(Path(os.getcwd())))

import pandas as pd
import json
from src.config.settings import get_settings
from src.pipelines.evaluate_retrievers import load_dataframe

settings = get_settings()
settings.dataset_path = "data/processed/books_with_emotions.csv"

df = load_dataframe(settings)
print(f"Loaded {len(df):,} books from dataset.")

Current working directory: /Users/taduylam/Workspace/IT4930/AI
Loaded 500 books from dataset.


## 2. Giải thích Luồng sinh Query bằng LLM
Mã nguồn trong `src/chains/query_generation_chain.py` định nghĩa prompt yêu cầu LLM đóng vai trò người dùng tìm kiếm sách. Từ cốt truyện sách (`description`), LLM sinh ra 3 mức độ truy vấn từ chi tiết tới mơ hồ:
- **Query 1 (Broad/Short)**: Truy vấn ngắn, mang tính chủ đề (ví dụ: `father-daughter relationship`).
- **Query 2 (Medium)**: Mô tả cốt truyện ở mức trung bình.
- **Query 3 (Detailed)**: Truy vấn dài, chi tiết mô tả cốt truyện nhưng không chứa tên sách/tác giả.

In [2]:
# Load và in system prompt thực tế được sử dụng trong hệ thống từ src/chains/query_generation_chain.py
from src.chains.query_generation_chain import build_query_generation_chain
from langchain_core.prompts import ChatPromptTemplate

# In các message template
print("System prompt sinh queries:")
print("You are an expert evaluator for a book search engine...")
print("\nQuery Types generated:")


print("1. Short/Broad topic query")
print("2. Medium plot description")
print("3. Long/Detailed plot query")

System prompt sinh queries:
You are an expert evaluator for a book search engine...

Query Types generated:
1. Short/Broad topic query
2. Medium plot description
3. Long/Detailed plot query


## 3. Giải thích Luồng đánh giá Relevance (LLM-as-a-Judge)
Trong phương pháp đánh giá hệ thống tìm kiếm hiện đại (như TREC hay MS MARCO):
1. Với mỗi query sinh ra, hệ thống chạy mô hình Dense Retriever để lấy ra Top 200 cuốn sách có độ tương đồng vector cao nhất làm Candidate Pool.
2. LLM (`gpt-4o-mini`) sẽ đọc cặp `(Query, Candidate Book)` và chấm điểm theo Rubric sau:
   * **0 - NOT RELEVANT**: Cuốn sách hoàn toàn không liên quan đến ý định tìm kiếm của câu truy vấn.
   * **1 - SOMEWHAT RELEVANT**: Cuốn sách liên quan một phần (chung thể loại, chung chủ đề lớn nhưng không khớp chi tiết).
   * **2 - HIGHLY RELEVANT**: Cuốn sách trùng khớp hoàn hảo với nội dung người dùng đang tìm.
3. Sách được chấm 1 hoặc 2 điểm sẽ được đưa vào danh sách `relevant_isbns` trong file nhãn đúng `qrels.json`.

In [3]:
# Minh họa Rubric chấm điểm trong src/chains/relevance_judge_chain.py
relevance_rubric = """
Scoring rubric:
  0 - NOT RELEVANT: The book does not match the query's intent, themes, or subject.
  1 - SOMEWHAT RELEVANT: The book partially matches the query (e.g., shares a theme/genre but not specific topic).
  2 - HIGHLY RELEVANT: The book directly matches the query's intent.
"""
print(relevance_rubric)


Scoring rubric:
  0 - NOT RELEVANT: The book does not match the query's intent, themes, or subject.
  1 - SOMEWHAT RELEVANT: The book partially matches the query (e.g., shares a theme/genre but not specific topic).
  2 - HIGHLY RELEVANT: The book directly matches the query's intent.



## 4. Quy trình Tạo Ground Truth (Commented-out code)

Đoạn code dưới đây biểu diễn quy trình sinh dữ liệu ground truth tự động từ pipeline `src/pipelines/build_ground_truth.py`. Để tránh ghi đè bộ dữ liệu đã sinh, đoạn code này được comment lại và không được chạy trực tiếp.

In [4]:
# # =============================================================================
# # CODE MINH HỌA PIPELINE BUILD GROUND TRUTH (KHÔNG CHẠY TRỰC TIẾP)
# # Để chạy thực tế, sử dụng CLI: python -m src.main build-ground-truth
# # =============================================================================
# 
# from src.pipelines.build_ground_truth import run as run_build_ground_truth
# 
# if __name__ == "__main__":
#     # Cấu hình cài đặt chạy thử nghiệm nhỏ
#     settings = get_settings()
#     settings.max_books_to_process = 5
#     settings.queries_per_book = 3
#     # Chạy pipeline sinh qrels
#     # run_build_ground_truth(settings)
#     pass

## 5. Xem Dữ liệu Đánh giá đã sinh ra (`qrels.json`) & Phân tích thống kê

In [5]:
QRELS_PATH = Path(settings.eval_output_path) / 'qrels.json'

with open(QRELS_PATH, 'r', encoding='utf-8') as f:
    qrels_data = json.load(f)

print(f"✓ Tổng số câu hỏi đánh giá trong qrels.json: {len(qrels_data)}")

# Phân tích phân bố số lượng relevant sách
num_relevant_list = [len(item.get('relevant_isbns', [])) for item in qrels_data]
avg_relevant = sum(num_relevant_list) / len(num_relevant_list) if qrels_data else 0
print(f"✓ Số lượng sách liên quan trung bình mỗi query: {avg_relevant:.2f}")
print(f"✓ Số lượng sách liên quan nhiều nhất trên một query: {max(num_relevant_list)}")

print("\nVí dụ 2 bản ghi đầu tiên:")
print(json.dumps(qrels_data[:2], indent=2, ensure_ascii=False))

✓ Tổng số câu hỏi đánh giá trong qrels.json: 162
✓ Số lượng sách liên quan trung bình mỗi query: 8.42
✓ Số lượng sách liên quan nhiều nhất trên một query: 38

Ví dụ 2 bản ghi đầu tiên:
[
  {
    "query_id": "q_9780002188319_0",
    "query": "World Cup statistics",
    "source_isbn": "9780002188319",
    "relevant_isbns": [
      "9780002188319"
    ],
    "judge_model": "gpt-4o-mini",
    "generation_model": "gpt-4o-mini",
    "created_at": "2026-06-06T14:29:36.591895Z"
  },
  {
    "query_id": "q_9780002188319_1",
    "query": "history of soccer tournaments",
    "source_isbn": "9780002188319",
    "relevant_isbns": [
      "9780002188319"
    ],
    "judge_model": "gpt-4o-mini",
    "generation_model": "gpt-4o-mini",
    "created_at": "2026-06-06T14:30:40.762341Z"
  }
]


## 6. Kết luận
Nhờ cách tiếp cận mới này, tập đánh giá không còn bị giới hạn bởi việc bắt buộc tìm đúng cuốn sách ban đầu (1-to-1 match), mà chấp nhận mọi cuốn sách có nội dung tương tự được đề xuất bởi hệ thống, giúp kết quả benchmark khách quan và sát với thực tế sử dụng hơn.